# Week 7 ? Dense retrieval and embedding comparison

Run from top to bottom with the project .venv kernel and Docker/Qdrant running. This notebook indexes 60 provisional records and compares BGE-M3 with multilingual-e5-base on five exploratory queries. Repeated upserts update stable IDs. No model fine-tuning or text generation is performed.


In [1]:
import os
import sys

# Set before importing Hugging Face libraries; keep certificate checks enabled.
os.environ["HF_HUB_DISABLE_XET"] = "1"
print("Python:", sys.executable)


Python: C:\Users\Roxi\llm-project-2026\.venv\Scripts\python.exe


1. Connect to local Qdrant

Start the existing container with `docker start llm-qdrant`. An empty collection list is expected only before the first ingestion.


In [2]:
from qdrant_client import QdrantClient

client = QdrantClient(url="http://localhost:6333")
print(client.get_collections())

collections=[CollectionDescription(name='week7_bge_m3'), CollectionDescription(name='week7_multilingual_e5_base')]


2. Load and validate the sample

Source and provisional approval metadata are preserved in Qdrant payloads. This sample is for local prototyping; the notebook does not approve the data.


In [3]:
import json
from pathlib import Path

project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "docs" / "timeline.md").is_file()
     and (path / "src").is_dir()),
    None
)
if project_root is None:
    raise RuntimeError("Open this notebook from inside the project folder.")

data_path = project_root / "data" / "processed" / "chunks.jsonl"
if not data_path.is_file():
    raise FileNotFoundError("Run python src/ingestion/clean.py from the project root first.")
with data_path.open(encoding="utf-8") as file:
    chunks = [json.loads(line) for line in file if line.strip()]
assert len(chunks) == 60, "This experiment expects the original 60-fragment sample."
assert len({chunk["id"] for chunk in chunks}) == len(chunks)
assert all(chunk["text"].strip() for chunk in chunks)
print("Num?r de fragmente:", len(chunks))
print("\nPrimul fragment:")
print(chunks[0]["text"])


Num?r de fragmente: 60

Primul fragment:
Rele-s, bade, frigurile
Da-s mai rele dragostele.
Maica din friguri mă scóte,
Dar din dragoste nu póte.


3. Load BGE-M3 and encode texts

The model revision is fixed for reproducibility. Downloads are cached outside Git. Do not restart the kernel while downloading.


In [4]:
from sentence_transformers import SentenceTransformer

BGE_MODEL_ID = "BAAI/bge-m3"
BGE_REVISION = "5617a9f61b028005a4858fdac845db406aefb181"
print("Loading BGE-M3 (CPU); first use may require a download...", flush=True)
model = SentenceTransformer(BGE_MODEL_ID, revision=BGE_REVISION, device="cpu")
print("Modelul BGE-M3 este ?nc?rcat.")


Loading BGE-M3 (CPU); first use may require a download...
Modelul BGE-M3 este ?nc?rcat.


Loading weights: 100%|##########| 391/391 [00:00<00:00, 3715.91it/s]


In [5]:
texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(
    texts,
    batch_size=2,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Dimensiunea rezultatului:", embeddings.shape)


Dimensiunea rezultatului: (60, 1024)


Batches: 100%|##########| 30/30 [01:36<00:00,  3.21s/it]


4. Create and populate the BGE-M3 collection

Use normalized dense vectors and cosine similarity. An incompatible existing collection raises an error; no collection is deleted.


In [6]:
from qdrant_client import QdrantClient, models

client = QdrantClient(url="http://localhost:6333")
collection_name = "week7_bge_m3"

if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=embeddings.shape[1],
            distance=models.Distance.COSINE
        )
    )

print("Colecția este pregătită:", collection_name)

Colecția este pregătită: week7_bge_m3


In [7]:
import uuid

points = [
    models.PointStruct(
        id=str(uuid.uuid5(uuid.NAMESPACE_URL, chunk["id"])),
        vector=embedding.tolist(),
        payload=chunk
    )
    for chunk, embedding in zip(chunks, embeddings)
]

client.upsert(
    collection_name=collection_name,
    points=points,
    wait=True
)

count = client.count(
    collection_name=collection_name,
    exact=True
).count

print("Fragmente salvate în Qdrant:", count)
assert count == len(chunks), "Unexpected collection size; inspect before continuing."


Fragmente salvate în Qdrant: 60


5. Demonstrate retrieval

Scores indicate similarity within a model; they are not correctness percentages.


In [8]:
query = "suferință din iubire și dor"

query_vector = model.encode(
    query,
    normalize_embeddings=True
)

results = client.query_points(
    collection_name=collection_name,
    query=query_vector.tolist(),
    limit=5,
    with_payload=True
).points

for position, result in enumerate(results, start=1):
    print(f"\n--- Rezultatul {position} | Scor: {result.score:.3f} ---")
    print("ID:", result.payload["id"])
    print("Mod:", result.payload["mode"])
    print(result.payload["text"])



--- Rezultatul 1 | Scor: 0.531 ---
ID: rap-original-014
Mod: rap
Nu am reflectoare, doar un bec pe coridor,
Dar pun în patru versuri ce mă doare și mi-e dor.
Când vocea se ridică, nu caut aplauze,
Ci un adevăr mai simplu între două scurte pauze.

--- Rezultatul 2 | Scor: 0.526 ---
ID: folk-jb1895-029
Mod: folk
Sărac plug cu șése boi,
Dragu’mi-i mie de voi!
Și mi-i drag de cin’ vĕ mână,
Că ține biciu ’ntr’o mână
Și strigă dela inimă!

--- Rezultatul 3 | Scor: 0.518 ---
ID: folk-jb1895-008
Mod: folk
Pân’ eram de nu iubiam,
Unde mă culcam, dormiam;
Dar acum, de când iubesc,
Nu pot să mă odihnesc.
Pe braț capu-mi sprijinesc
Și tot la mândra gândesc.

--- Rezultatul 4 | Scor: 0.515 ---
ID: folk-jb1895-004
Mod: folk
De urît m’aș duce’n țară,
Dragostea mă’ntórnă éră;
De urît m’aș duce’n lume,
Dragostea capu mi-l pune!

--- Rezultatul 5 | Scor: 0.498 ---
ID: rap-original-018
Mod: rap
În spatele scenei, se-aude-un murmur greu,
Îmi bate inima mai tare decât ritmul meu.
Când ies spre lumină, nu 

6. Run the exploratory BGE-M3 query set

Both models receive the same five queries and the same corpus. This is not a locked benchmark.


In [9]:
test_queries = [
    "suferință din iubire și dor",
    "mama și familia",
    "viața la sat și munca",
    "muzică, scenă și emoțiile artistului",
    "dragoste care nu mă lasă să dorm"
]

print("Număr de căutări pentru comparație:", len(test_queries))

Număr de căutări pentru comparație: 5


In [10]:
bge_results = {}

for query in test_queries:
    vector = model.encode(query, normalize_embeddings=True)

    hits = client.query_points(
        collection_name=collection_name,
        query=vector.tolist(),
        limit=5,
        with_payload=True
    ).points

    bge_results[query] = [
        {
            "id": hit.payload["id"],
            "text": hit.payload["text"],
            "score": hit.score
        }
        for hit in hits
    ]

    print(f"\nCăutare: {query}")
    for position, hit in enumerate(hits, start=1):
        print(f"{position}. {hit.payload['id']} — {hit.score:.3f}")

print("\nRezultatele BGE-M3 sunt păstrate în memorie.")



Căutare: suferință din iubire și dor
1. rap-original-014 — 0.531
2. folk-jb1895-029 — 0.526
3. folk-jb1895-008 — 0.518
4. folk-jb1895-004 — 0.515
5. rap-original-018 — 0.498

Căutare: mama și familia
1. rap-original-011 — 0.487
2. folk-jb1895-026 — 0.447
3. folk-jb1895-018 — 0.445
4. folk-jb1895-002 — 0.438
5. folk-jb1895-024 — 0.434

Căutare: viața la sat și munca
1. rap-original-009 — 0.571
2. folk-jb1895-024 — 0.553
3. rap-original-010 — 0.507
4. folk-jb1895-030 — 0.503
5. rap-original-016 — 0.491

Căutare: muzică, scenă și emoțiile artistului
1. rap-original-018 — 0.643
2. rap-original-010 — 0.534
3. rap-original-014 — 0.519
4. rap-original-005 — 0.513
5. rap-original-001 — 0.506

Căutare: dragoste care nu mă lasă să dorm
1. folk-jb1895-008 — 0.611
2. folk-jb1895-012 — 0.570
3. folk-jb1895-009 — 0.527
4. rap-original-019 — 0.515
5. folk-jb1895-004 — 0.503

Rezultatele BGE-M3 sunt păstrate în memorie.


In [11]:
results_dir = project_root / "eval" / "week7"
results_dir.mkdir(parents=True, exist_ok=True)

results_path = results_dir / "bge_m3_results.json"

with results_path.open("w", encoding="utf-8") as file:
    json.dump(bge_results, file, ensure_ascii=False, indent=2)

print("Rezultate salvate în:", results_path)

Rezultate salvate în: C:\Users\Roxi\llm-project-2026\eval\week7\bge_m3_results.json


7. Download and load multilingual-e5-base

Download one file at a time, reusing cached files. Then load the local snapshot. A stalled transfer requires diagnosis rather than waiting indefinitely.


In [12]:
from huggingface_hub import snapshot_download
from tqdm import tqdm

E5_MODEL_ID = "intfloat/multilingual-e5-base"
E5_REVISION = "d128750597153bb5987e10b1c3493a34e5a4502a"
print("Downloading E5 files (or reusing the local cache)...", flush=True)
e5_path = snapshot_download(
    repo_id=E5_MODEL_ID,
    revision=E5_REVISION,
    allow_patterns=["*.json", "*.txt", "*.model", "model.safetensors"],
    max_workers=1,
    tqdm_class=tqdm
)
print("Desc?rcare terminat?:", e5_path)


Desc?rcare terminat?: C:\Users\Roxi\.cache\huggingface\hub\models--intfloat--multilingual-e5-base\snapshots\d128750597153bb5987e10b1c3493a34e5a4502a


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files: 100%|##########| 14/14 [00:00<00:00, 442.17it/s]
Download complete:           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B                         

In [13]:
e5_model = SentenceTransformer(
    e5_path,
    device="cpu",
    local_files_only=True
)

print("Modelul E5 este încărcat.")

Modelul E5 este încărcat.


Loading weights: 100%|##########| 199/199 [00:00<00:00, 219.24it/s]


8. Encode and index the same sample with E5

E5 requires `passage: ` for corpus texts and `query: ` for retrieval queries, including Romanian. Its 768-dimensional vectors use a separate collection.


In [14]:
e5_texts = ["passage: " + chunk["text"] for chunk in chunks]

e5_embeddings = e5_model.encode(
    e5_texts,
    batch_size=2,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Dimensiunea rezultatului:", e5_embeddings.shape)

Dimensiunea rezultatului: (60, 768)


Batches: 100%|##########| 30/30 [00:28<00:00,  1.05it/s]


In [15]:
e5_collection_name = "week7_multilingual_e5_base"

if not client.collection_exists(e5_collection_name):
    client.create_collection(
        collection_name=e5_collection_name,
        vectors_config=models.VectorParams(
            size=e5_embeddings.shape[1],
            distance=models.Distance.COSINE
        )
    )

config = client.get_collection(e5_collection_name).config.params.vectors
assert config.size == e5_embeddings.shape[1], "Existing collection has incompatible dimensions."
assert config.distance == models.Distance.COSINE, "Expected cosine distance."
assert len(chunks) == len(e5_embeddings)

e5_points = [
    models.PointStruct(
        id=str(uuid.uuid5(uuid.NAMESPACE_URL, chunk["id"])),
        vector=embedding.tolist(),
        payload=chunk
    )
    for chunk, embedding in zip(chunks, e5_embeddings)
]

client.upsert(
    collection_name=e5_collection_name,
    points=e5_points,
    wait=True
)

count = client.count(
    collection_name=e5_collection_name,
    exact=True
).count

print("Fragmente salvate pentru E5:", count)
assert count == len(chunks), "Unexpected collection size; inspect before continuing."


Fragmente salvate pentru E5: 60


9. Run and save E5 retrieval

Compare retrieved text and rank, not raw similarity values across models.


In [16]:
e5_results = {}

for query in test_queries:
    vector = e5_model.encode(
        "query: " + query,
        normalize_embeddings=True
    )

    hits = client.query_points(
        collection_name=e5_collection_name,
        query=vector.tolist(),
        limit=5,
        with_payload=True
    ).points

    e5_results[query] = [
        {
            "id": hit.payload["id"],
            "text": hit.payload["text"],
            "score": hit.score
        }
        for hit in hits
    ]

    print(f"\nCăutare: {query}")
    for position, hit in enumerate(hits, start=1):
        print(f"{position}. {hit.payload['id']} — {hit.score:.3f}")

with (results_dir / "e5_base_results.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(e5_results, file, ensure_ascii=False, indent=2)

print("\nRezultatele E5 au fost salvate.")




Căutare: suferință din iubire și dor
1. rap-original-019 — 0.833
2. folk-jb1895-008 — 0.833
3. folk-jb1895-004 — 0.832
4. folk-jb1895-012 — 0.831
5. folk-jb1895-001 — 0.830

Căutare: mama și familia
1. rap-original-016 — 0.827
2. rap-original-013 — 0.821
3. folk-jb1895-009 — 0.820
4. rap-original-011 — 0.819
5. folk-jb1895-012 — 0.817

Căutare: viața la sat și munca
1. rap-original-012 — 0.833
2. rap-original-009 — 0.833
3. rap-original-016 — 0.831
4. rap-original-001 — 0.827
5. rap-original-010 — 0.826

Căutare: muzică, scenă și emoțiile artistului
1. rap-original-018 — 0.847
2. rap-original-012 — 0.837
3. rap-original-014 — 0.834
4. rap-original-005 — 0.831
5. rap-original-016 — 0.826

Căutare: dragoste care nu mă lasă să dorm
1. folk-jb1895-008 — 0.843
2. folk-jb1895-001 — 0.841
3. folk-jb1895-009 — 0.838
4. folk-jb1895-012 — 0.835
5. folk-jb1895-013 — 0.832

Rezultatele E5 au fost salvate.


10. Inspect results for each query

The next five cells retain the inspected texts for review.


In [17]:
query = test_queries[0]

print("CĂUTARE:", query)

for model_name, results in [
    ("BGE-M3", bge_results),
    ("E5-base", e5_results)
]:
    print(f"\n========== {model_name} ==========")

    for position, result in enumerate(results[query], start=1):
        print(f"\n{position}. {result['id']}")
        print(result["text"])

CĂUTARE: suferință din iubire și dor

========== BGE-M3 ==========

1. rap-original-014
Nu am reflectoare, doar un bec pe coridor,
Dar pun în patru versuri ce mă doare și mi-e dor.
Când vocea se ridică, nu caut aplauze,
Ci un adevăr mai simplu între două scurte pauze.

2. folk-jb1895-029
Sărac plug cu șése boi,
Dragu’mi-i mie de voi!
Și mi-i drag de cin’ vĕ mână,
Că ține biciu ’ntr’o mână
Și strigă dela inimă!

3. folk-jb1895-008
Pân’ eram de nu iubiam,
Unde mă culcam, dormiam;
Dar acum, de când iubesc,
Nu pot să mă odihnesc.
Pe braț capu-mi sprijinesc
Și tot la mândra gândesc.

4. folk-jb1895-004
De urît m’aș duce’n țară,
Dragostea mă’ntórnă éră;
De urît m’aș duce’n lume,
Dragostea capu mi-l pune!

5. rap-original-018
În spatele scenei, se-aude-un murmur greu,
Îmi bate inima mai tare decât ritmul meu.
Când ies spre lumină, nu joc un rol perfect,
Ci spun ce-am strâns în mine, simplu, viu și direct.

========== E5-base ==========

1. rap-original-019
Autobuzul urcă, e frig și seara vine

In [18]:
query = test_queries[1]

print("CĂUTARE:", query)

for model_name, results in [
    ("BGE-M3", bge_results),
    ("E5-base", e5_results)
]:
    print(f"\n========== {model_name} ==========")

    for position, result in enumerate(results[query], start=1):
        print(f"\n{position}. {result['id']}")
        print(result["text"])

CĂUTARE: mama și familia

========== BGE-M3 ==========

1. rap-original-011
Am foile pe masă și-un ceas care aleargă,
Respir, citesc cerința, las teama să se șteargă.
Un pas după alt pas, răspunsul se desface,
Iar mintea mea găsește un pic de loc și pace.

2. folk-jb1895-026
Tătăișă, draga mea,
Nu te tare supĕra,
Că mândră ca dumneata
Nu-i nime, câtu-i lumea;
Și câtu-i satul de mare
Nu-i casă c’a dumitale:
La ferești
Cu flori domnesci!

3. folk-jb1895-018
Mândra mea, mândră Mariă,
Scrisă-i fața ta ’n hârtiă;
Mândra mea, mândră Iónă,
Scrisă-i fața ta ’n icónă!

4. folk-jb1895-002
Zis’a maica, că mă scóte
Dela câte, dela tóte.
Dela doue nu mă póte:
Dela mândră, dela morte!

5. folk-jb1895-024
Fóie verde de sasăŭ
Țu, țu, țu, murguțul meŭ!
Să întrăm în sat cu sóre,
C’am o mândră ca ș’o flóre
Și ne-aștéptă cu mâncare.
Acolo dac’ om sosi,
Ea frumos ne-a omeni;
Ție iarbă și ovĕs,
Mie gură și scoverḑi!

========== E5-base ==========

1. rap-original-016
Trec pe lângă case, cu vântul în hanorac

In [19]:
query = test_queries[2]

print("CĂUTARE:", query)

for model_name, results in [
    ("BGE-M3", bge_results),
    ("E5-base", e5_results)
]:
    print(f"\n========== {model_name} ==========")

    for position, result in enumerate(results[query], start=1):
        print(f"\n{position}. {result['id']}")
        print(result["text"])

CĂUTARE: viața la sat și munca

========== BGE-M3 ==========

1. rap-original-009
În sat se-aude clopotul când soarele coboară,
Bătrânii spun istorii pe prispa de afară.
Le port prin orașe fără să le schimb rostul,
Că rădăcina ține viu și ramul, și adăpostul.

2. folk-jb1895-024
Fóie verde de sasăŭ
Țu, țu, țu, murguțul meŭ!
Să întrăm în sat cu sóre,
C’am o mândră ca ș’o flóre
Și ne-aștéptă cu mâncare.
Acolo dac’ om sosi,
Ea frumos ne-a omeni;
Ție iarbă și ovĕs,
Mie gură și scoverḑi!

3. rap-original-010
Ciocanul bate ritmul, o roată se rotește,
Un om repară lucruri și-n liniște zâmbește.
Mă uit la mâini muncite, la timpul lor cinstit,
Și scriu despre răbdarea cu care-au construit.

4. folk-jb1895-030
Mult mĕ mustră măicuța
Să las séra ulița,
Dar eŭ nu o pociŭ lăsare,
Că satul câtu-i de mare
Nime bădiț ca min’ n’are,
Și câtu-i satu mereŭ
Nu-i bădiță ca al meŭ.
Ba mai e un brad în munte
Cu crăci verḑi, cu verde frunte;
Dar nici bradul cel fălos
Nu-i ca badiŭ de frumos,
Că bradu mi-e clen

In [20]:
query = test_queries[3]

print("CĂUTARE:", query)

for model_name, results in [
    ("BGE-M3", bge_results),
    ("E5-base", e5_results)
]:
    print(f"\n========== {model_name} ==========")

    for position, result in enumerate(results[query], start=1):
        print(f"\n{position}. {result['id']}")
        print(result["text"])

CĂUTARE: muzică, scenă și emoțiile artistului

========== BGE-M3 ==========

1. rap-original-018
În spatele scenei, se-aude-un murmur greu,
Îmi bate inima mai tare decât ritmul meu.
Când ies spre lumină, nu joc un rol perfect,
Ci spun ce-am strâns în mine, simplu, viu și direct.

2. rap-original-010
Ciocanul bate ritmul, o roată se rotește,
Un om repară lucruri și-n liniște zâmbește.
Mă uit la mâini muncite, la timpul lor cinstit,
Și scriu despre răbdarea cu care-au construit.

3. rap-original-014
Nu am reflectoare, doar un bec pe coridor,
Dar pun în patru versuri ce mă doare și mi-e dor.
Când vocea se ridică, nu caut aplauze,
Ci un adevăr mai simplu între două scurte pauze.

4. rap-original-005
Pe podul peste Someș, luminile se-adună,
Sub pași curge un cântec, deși nimeni nu-l îngână.
Am întrebări în minte și răspunsuri încă mici,
Dar fiecare pas mă ține între ai mei, aici.

5. rap-original-001
Pe strada udă, pașii mei bat tactul în asfalt,
Clujul aprinde geamuri, iar gândul urcă-nalt

In [21]:
query = test_queries[4]

print("CĂUTARE:", query)

for model_name, results in [
    ("BGE-M3", bge_results),
    ("E5-base", e5_results)
]:
    print(f"\n========== {model_name} ==========")

    for position, result in enumerate(results[query], start=1):
        print(f"\n{position}. {result['id']}")
        print(result["text"])

CĂUTARE: dragoste care nu mă lasă să dorm

========== BGE-M3 ==========

1. folk-jb1895-008
Pân’ eram de nu iubiam,
Unde mă culcam, dormiam;
Dar acum, de când iubesc,
Nu pot să mă odihnesc.
Pe braț capu-mi sprijinesc
Și tot la mândra gândesc.

2. folk-jb1895-012
Pentru tine, rujă plină,
Nici n’am somn, nici n’am odihnă;
Pentru tine, rujă créță,
Nici n’am somn, nici n’am viață!

3. folk-jb1895-009
Însura-m’aș, să-mi fac șură,
Dar inima nu mă’ndură;
Însura-m’aș, să-mi fac casă,
Dar inima nu mă lasă!

4. rap-original-019
Autobuzul urcă, e frig și seara vine,
În telefon am mesaje de la oamenii dragi mie.
La capătul călătoriei mă așteaptă un pridvor,
Și toate grijile se-așază mai ușor.

5. folk-jb1895-004
De urît m’aș duce’n țară,
Dragostea mă’ntórnă éră;
De urît m’aș duce’n lume,
Dragostea capu mi-l pune!

========== E5-base ==========

1. folk-jb1895-008
Pân’ eram de nu iubiam,
Unde mă culcam, dormiam;
Dar acum, de când iubesc,
Nu pot să mă odihnesc.
Pe braț capu-mi sprijinesc
Și tot la m

11. Record proposed relevance labels

Labels were proposed after inspecting pooled top-5 results. They require team review and do not exhaustively label the corpus. They support exploratory Precision@5, not corpus-wide recall.


In [22]:
evaluation = {
    "status": "preliminary_pending_team_review",
    "method": "Manual relevance assessment of the top 5 results",
    "limitations": [
        "Only 5 exploratory queries",
        "Labels assigned after inspecting retrieval results",
        "Team review required"
    ],
    "queries": [
        {
            "query": test_queries[0],
            "criterion": "Explicit romantic suffering or longing",
            "relevant_ids": [
                "folk-jb1895-008", "folk-jb1895-004",
                "folk-jb1895-012", "folk-jb1895-001"
            ]
        },
        {
            "query": test_queries[1],
            "criterion": "Explicit mother, relatives or family formation",
            "relevant_ids": ["folk-jb1895-002", "folk-jb1895-009"]
        },
        {
            "query": test_queries[2],
            "criterion": "Rural life or work as a substantive theme",
            "relevant_ids": [
                "rap-original-009", "folk-jb1895-024",
                "rap-original-010"
            ]
        },
        {
            "query": test_queries[3],
            "criterion": "Artistic performance and associated emotions",
            "relevant_ids": ["rap-original-018", "rap-original-014"]
        },
        {
            "query": test_queries[4],
            "criterion": "Explicit sleeplessness caused by romantic love",
            "relevant_ids": ["folk-jb1895-008", "folk-jb1895-012"]
        }
    ]
}

evaluation_path = results_dir / "relevance_review.json"

with evaluation_path.open("w", encoding="utf-8") as file:
    json.dump(evaluation, file, ensure_ascii=False, indent=2)

print("Evaluare salvată în:", evaluation_path)


Evaluare salvată în: C:\Users\Roxi\llm-project-2026\eval\week7\relevance_review.json


12. Compute and save exploratory Precision@5

Precision@5 is the number of proposed relevant results among five retrieved results, divided by five. Report the mean over the five queries. A one-result difference does not establish an overall winner.


In [23]:
summary = {}

for model_name, results in [
    ("BGE-M3", bge_results),
    ("E5-base", e5_results)
]:
    scores = []

    print(f"\n{model_name}")

    for item in evaluation["queries"]:
        query = item["query"]
        relevant_ids = set(item["relevant_ids"])
        top5 = results[query][:5]

        assert len(top5) == 5, "Expected exactly five retrieved results."

        relevant_count = sum(
            result["id"] in relevant_ids
            for result in top5
        )

        precision = relevant_count / 5
        scores.append(precision)

        print(f"{query}: {precision:.0%}")

    average = sum(scores) / len(scores)

    summary[model_name] = {
        "precision_at_5_per_query": scores,
        "mean_precision_at_5": average
    }

    print(f"Media Precision@5: {average:.0%}")

with (results_dir / "metrics.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(summary, file, ensure_ascii=False, indent=2)

print("\nMetricile au fost salvate.")


BGE-M3
suferință din iubire și dor: 40%
mama și familia: 20%
viața la sat și munca: 60%
muzică, scenă și emoțiile artistului: 40%
dragoste care nu mă lasă să dorm: 40%
Media Precision@5: 40%

E5-base
suferință din iubire și dor: 80%
mama și familia: 20%
viața la sat și munca: 40%
muzică, scenă și emoțiile artistului: 40%
dragoste care nu mă lasă să dorm: 40%
Media Precision@5: 44%

Metricile au fost salvate.
